In [ ]:
import numpy as np
import shutil
import random
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import torchvision
from torchvision.transforms import v2
from torchvision.models import (resnet50, efficientnet_b0, 
                                mobilenet_v3_small, densenet121)
from torchvision.models import resnet18
from torch.utils.data import DataLoader
from torch.utils.data import DataLoader, Subset
import os
import torch.nn.functional as F
import torch
import torch.backends.cudnn as cudnn

#"dataset_path"is the file address of the corresponding dataset, you need change it yourself
dataset_params = {
        'CIFAR10': {
            'path' : "dataset_path",
            'epochs_train': 100,
            'batch_size': 512,
            'mean_normalize': (0.4914, 0.4822, 0.4465),
            'std_normalize': (0.2023, 0.1994, 0.2010),
            'num_classes': 10,
            'Tmin_knn': 1,
            'Tmax_knn': 3,
            'k_knn': 20,
            'p_knn': 1/2,
            'Tmin_C': 1,
            'Tmax_C': 3,
            'Tmin_f': 1,
            'Tmax_f': 3,
            'p_f': 2.5,
            'ad_C': 2
        },
        'CIFAR100': {
            'path' : "dataset_path",
            'epochs_train': 300,
            'batch_size': 512,
            'mean_normalize': (0.5071, 0.4865, 0.4409),
            'std_normalize': (0.2673, 0.2564, 0.2762),
            'num_classes': 100,
            'Tmin_knn': 1,
            'Tmax_knn': 5,
            'k_knn': 200,
            'p_knn': 1/2,
            'Tmin_C': 1,
            'Tmax_C': 5,
            'Tmin_f': 1,
            'Tmax_f': 5,
            'p_f': 2.5,
            'ad_C': 20
        },
        'TinyImageNet':{
            'path' : "dataset_path",
            'epochs_train': 300,
            'batch_size': 256,
            'mean_normalize': (0.485, 0.456, 0.406),
            'std_normalize': (0.229, 0.224, 0.225),
            'num_classes': 200,
            'Tmin_knn': 1,
            'Tmax_knn': 5,
            'k_knn': 400,
            'p_knn': 1/2,
            'Tmin_C': 1,
            'Tmax_C': 5,
            'Tmin_f': 1,
            'Tmax_f': 5,
            'p_f': 2.5
            ,'ad_C': 40
        },
    }

os.makedirs('fig', exist_ok=True)
device = 'cuda'
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [ ]:
#Function definition
def reorganize_tinyimagenet_val_temp(original_val_root, processed_val_root):
    images_dir = os.path.join(original_val_root, 'images')
    annotations_file = os.path.join(original_val_root, 'val_annotations.txt')
    
    if not os.path.exists(annotations_file):
        raise FileNotFoundError(f"Cannot find {annotations_file}")
    if not os.path.exists(images_dir):
        raise FileNotFoundError(f"Cannot find {images_dir}")
    if os.path.exists(processed_val_root):
        subdirs = [d for d in os.listdir(processed_val_root) if os.path.isdir(os.path.join(processed_val_root, d))]
        if len(subdirs) > 0:
            print(f"Using existing processed TinyImageNet val folder: {processed_val_root}")
            return
    print("Creating temporary processed TinyImageNet val folder (this may take a minute)...")
    os.makedirs(processed_val_root, exist_ok=True)
    with open(annotations_file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            image_name, class_name = parts[0], parts[1]
            
            class_dir = os.path.join(processed_val_root, class_name)
            os.makedirs(class_dir, exist_ok=True)
            
            src = os.path.join(images_dir, image_name)
            dst = os.path.join(class_dir, image_name)
            
            if os.path.exists(src):
                shutil.copy2(src, dst)
    
    print(f"TinyImageNet val processed and copied to: {processed_val_root}")

def create_model_train(model_name, num_classes, dataset_name):
    model_name = model_name.lower()
    dataset_name = dataset_name.lower()
    if model_name == "resnet50":
        net = resnet50(weights=None)
        net.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        net.maxpool = nn.Identity() 
        net.fc = nn.Linear(net.fc.in_features, num_classes)
    elif model_name == "efficientnetb0":
        net = efficientnet_b0(weights=None)
        net.features[0][0].stride = (1, 1)
        net.classifier[1] = nn.Linear(net.classifier[1].in_features, num_classes)
    elif model_name == "mobilenetv3small":
        net = mobilenet_v3_small(weights=None)
        net.features[0][0].stride = (1, 1)
        net.classifier[3] = nn.Linear(net.classifier[3].in_features, num_classes)
    elif model_name == "densenet121":
        net = densenet121(weights=None)
        net.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        net.features.pool0 = nn.Identity()
        net.classifier = nn.Linear(net.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Model {model_name} is not supported.")
    return net

def entropy_knn(U_pca, y, k, n_class):
    n = len(U_pca)
    dist_matrix = torch.cdist(U_pca, U_pca, p = 2)  # (n_samples, n_samples)    
    _, indices = torch.topk(dist_matrix, k=k+1, largest=False, dim=1)    
    knn_labels = y[indices[:, 1:]]
    label_freq_matrix = torch.zeros((n, n_class), device=U_pca.device)
    for i in range(n_class):
        label_freq_matrix[:, i] = (knn_labels == i).sum(dim=1).float()
    label_freq = label_freq_matrix / k
    entropy_values = -torch.sum(label_freq * torch.log(label_freq + 1e-10), dim=1)
    return entropy_values

def bin_thresholds(en_t2_cal, Tmax, Tmin, p):
    en_t2_min, _ = en_t2_cal.min(dim=1)
    en_t2_max, _ = en_t2_cal.max(dim=1)
    if Tmax <=Tmin:
        print("error,autocorrection")
        Tmax = Tmin + 1
    L = Tmax - Tmin + 1
    weigh_bin = torch.tensor([((i - 1) / (L - 1)) ** p for i in range(1, L + 1)]).to(en_t2_cal.device)
    bin = en_t2_min.unsqueeze(1) + torch.outer(en_t2_max-en_t2_min,weigh_bin)
    return bin

def bin_f_thresholds(en_t2_cal, n, Tmax, Tmin, p):
    m = en_t2_cal.shape[0]
    en_t2_min = torch.zeros((m,), device=en_t2_cal.device)  
    en_t2_max = torch.full((m,), np.log(n), device=en_t2_cal.device, dtype=torch.float)
    if Tmax <=Tmin:
        print("error,autocorrection")
        Tmax = Tmin + 1
    L = Tmax - Tmin + 1
    weigh_bin = torch.tensor([((i - 1) / (L - 1)) ** p for i in range(1, L + 1)]).to(en_t2_cal.device)
    bin = en_t2_min.unsqueeze(1) + torch.outer(en_t2_max-en_t2_min,weigh_bin)
    return bin

def T_ad(en_t2, Tmax, Tmin, p, bin):
    comparison = en_t2.unsqueeze(2) >= bin.unsqueeze(1)
    T = comparison.sum(dim=2).int()
    Tmin_tensor = Tmin * torch.ones_like(T).to(en_t2.device)
    T = T + Tmin_tensor - 1
    return T

def fh(t, w, eif):
    if t.shape == w.shape:
        result = torch.where(t >= w, 
                             1 - eif / (t - w + 1), 
                             (eif / w) * t)
    else:
        raise ValueError("t and w must have the same shape")
    return result

def gather_by_row(A, B):
    if A.dim() != 2 or B.dim() != 2:
        raise ValueError("A and B must be two-dimensional tensors")
    n, m = A.shape
    p, n_b = B.shape
    if n_b != n:
        raise ValueError(f"A and B must be two-dimensional tensors (got {n_b}, expected {n})")
    if B.dtype != torch.long:
        B = B.long()
    row_idx = torch.arange(n, device=A.device).view(1, n).expand(p, n)
    col_idx = B
    C = A[row_idx, col_idx]
    return C

def a_an(or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp):
    print(f"real mispro: {re_pronp:.3g}")
    or_gap_LOOnp = np.mean(np.abs(or_alpha_LOOnp - re_pronp))
    or_median_LOOnp = np.median(or_alpha_LOOnp)
    or_std_LOOnp = np.std(or_alpha_LOOnp)
    or_mse = np.mean((or_alpha_LOOnp - or_alpha_realnp) ** 2)
    tr_gap_LOOnp = np.mean(np.abs(tr_alpha_LOOnp - re_pronp))
    tr_median_LOOnp = np.median(tr_alpha_LOOnp)
    tr_std_LOOnp = np.std(tr_alpha_LOOnp)
    tr_mse = np.mean((tr_alpha_LOOnp - tr_alpha_realnp) ** 2)
    
    print(f"or MED of LOO: {or_median_LOOnp:.3g}")
    print(f"or STD of LOO: {or_std_LOOnp:.3g}")
    print(f"or MSE between LOO and real: {or_mse:.3g}")
    print(f"or GAP of LOO: {or_gap_LOOnp:.3g}")
    print(f"tr MED of LOO: {tr_median_LOOnp:.3g}")
    print(f"tr STD of LOO: {tr_std_LOOnp:.3g}")
    print(f"tr MSE between LOO and real: {tr_mse:.3g}")
    print(f"tr GAP of LOO: {tr_gap_LOOnp:.3g}\n")
    
def select_model(model_name, dataset_name, device):
    params = dataset_params[dataset_name]
    n_class = params['num_classes']
    net = create_model_train(model_name,n_class,dataset_name)
    weights_path = "weights/"+model_name+"_"+dataset_name
    if device == 'cuda':
        net = torch.nn.DataParallel(net)
        cudnn.benchmark = True
    state_dict = torch.load(weights_path, map_location=device)
    if isinstance(net, torch.nn.DataParallel):
        if not any(k.startswith('module.') for k in state_dict.keys()):
            state_dict = {'module.' + k: v for k, v in state_dict.items()}
    else:
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    net.load_state_dict(state_dict, strict=False)
    print("Model weights loaded successfully!")
    return net

def select_dataset(dataset_name):
    params = dataset_params[dataset_name]
    dataset_path = params['path']
    transform_test = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(params['mean_normalize'], params['std_normalize']),
        ])
    if dataset_name == 'CIFAR10':
        testset = torchvision.datasets.CIFAR10(
            root=dataset_path, train=False, download=False, transform=transform_test)
    elif dataset_name == 'CIFAR100':
        testset = torchvision.datasets.CIFAR100(
            root=dataset_path, train=False, download=False, transform=transform_test)
    elif dataset_name == 'TinyImageNet':
        original_val_root = os.path.join(dataset_path, 'val')
        processed_val_root = './tinyimagenet_val_processed'
        reorganize_tinyimagenet_val_temp(original_val_root, processed_val_root)
        testset = torchvision.datasets.ImageFolder(root=processed_val_root, transform=transform_test)
    else:
        raise ValueError(f"Dataset {dataset_name} is not supported.")
    img_0, _ = testset[0]
    C_data, H_data, W_data = img_0.shape 
    return testset, C_data, H_data, W_data

def funS_t3(net, testset, n_calibration, n_pici, n_ml, n_class, C_data, H_data, W_data, epoch, device):
    memory_idx = torch.empty(n_pici, n_ml, n_calibration + 1, dtype=torch.long, device=device)

    S_t3_calibration = torch.empty(epoch, n_calibration, n_class).to(device)
    S_t3_point_test = torch.empty(epoch, 1, n_class).to(device)
    y_all = torch.empty(epoch, n_calibration + 1).to(device)
    net.eval()
    with torch.no_grad():
        X_test_all, y_test_all = next(iter(DataLoader(testset, batch_size=len(testset))))
        X_test_all, y_test_all = X_test_all.to(device), y_test_all.to(device)
        for i in range(n_pici):
            idx = torch.randint(0, len(X_test_all), (n_ml, n_calibration + 1), device=device) # (n_ml, n_calibration + 1)
            memory_idx[i] = idx
            X_idx = X_test_all[idx]  # (n_ml, n_calibration + 1, channels, height, width)
            X_idx = X_idx.view(-1, C_data, H_data, W_data)
            y_idx = y_test_all[idx]  # (n_ml, n_calibration + 1)
            logits_idx = net(X_idx)  # (n_ml, n_calibration + 1, n_class)
            logits_idx = logits_idx.view(n_ml, n_calibration + 1, n_class)
            logits_point_test_idx = logits_idx[:, 0:1, :]  # (n_ml, 1, n_class)
            logits_calibration_idx = logits_idx[:, 1:, :]  # (n_ml, n_calibration, n_class)
            S_t3_calibration_idx = -F.log_softmax(logits_calibration_idx, dim=2)  # (n_ml, n_calibration, n_class)
            S_t3_point_test_idx = -F.log_softmax(logits_point_test_idx, dim=2)  # (n_ml, 1, n_class)  
            start_idx = i * n_ml
            end_idx = (i + 1) * n_ml
            S_t3_calibration[start_idx:end_idx] = S_t3_calibration_idx
            S_t3_point_test[start_idx:end_idx] = S_t3_point_test_idx
            y_all[start_idx:end_idx] = y_idx
    y_all = y_all.long()
    return S_t3_calibration, S_t3_point_test, y_all, memory_idx

def funr_t3(S_t3_calibration, S_t3_point_test):
    r_t3_calibration = torch.argsort(torch.argsort(S_t3_calibration, dim=2), dim=2) + 1
    r_t3_point_test = torch.argsort(torch.argsort(S_t3_point_test, dim=2), dim=2) + 1
    return r_t3_calibration, r_t3_point_test

def funture_rS(S_t3_calibration, S_t3_point_test, r_t3_calibration, r_t3_point_test, y_all):
    y_point_test = y_all[:, 0]  # (epoch,)
    y_calibration = y_all[:, 1:]  # (epoch, n_calibration)

    true_S_cal = torch.gather(S_t3_calibration, 2, y_calibration.unsqueeze(2)).squeeze(2)  # (epoch, n_calibration)
    true_S_test = torch.gather(S_t3_point_test, 2, y_point_test.unsqueeze(1).unsqueeze(2)).squeeze()  # (epoch,)
    true_r_cal = torch.gather(r_t3_calibration, 2, y_calibration.unsqueeze(2)).squeeze(2)  # (epoch, n_calibration)
    true_r_test = torch.gather(r_t3_point_test, 2, y_point_test.unsqueeze(1).unsqueeze(2)).squeeze()  # (epoch,) (epoch,)
    sorted_S_t3_cal, _ = torch.sort(S_t3_calibration, dim=2, descending=False)
    sorted_S_t3_test, _ = torch.sort(S_t3_point_test, dim=2, descending=False)

    return true_S_cal, true_S_test, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test

def row_entropy(T, num_classes=None):
    if num_classes is None:
        num_classes = int(T.max()) + 1
    one_hot = torch.nn.functional.one_hot(T, num_classes=num_classes)
    p = one_hot.float().mean(dim=1)
    eps = 1e-12
    H = -(p * (p + eps).log()).sum(dim=1)
    return H

def one_complex_ada_size(dist, i_f, y_js, y_cal, k, T_max, T_min, p):
    if y_js == -1:
        labels = y_cal      
        if i_f == -1:
            dist_correct = dist[:, 1:]
            i_f = 0
        else:
            dist_correct = torch.cat([dist[:, :i_f], dist[:, i_f+1:]],dim=1)           
    else:
        y_js = torch.tensor([y_js], device=y_cal.device)
        labels = torch.cat([y_js, y_cal])  
        if i_f == len(y_cal):
            raise ValueError(f"Attention, you are wrong: D_(n+1)^y undefined")  
        dist_correct = dist                                
    KN = torch.argsort(dist_correct, dim=1)[:, :k]  
    distribution = labels[KN]              
    en = row_entropy(distribution)    
    en = en.unsqueeze(0)     
    bins = bin_thresholds(en, T_max, T_min, p)
    T = T_ad(en[:,i_f].unsqueeze(1), T_max, T_min, p, bins)
    return T

def many_complex_ada_size(dist_all, dist_cal, epoch_ycal, n_class, n_calibration, k, Tmax, Tmin, p):
    js_y_all = torch.tensor(range(n_class), device=device)
    T_epoch = torch.zeros(n_class, n_calibration, device=device)
    T_epoch_LOO = torch.zeros(1, n_calibration, device=device)
    T_epoch_pointtest = one_complex_ada_size(dist_all, -1, -1, epoch_ycal, k, Tmax, Tmin, p)
    T_epoch_pointtest = T_epoch_pointtest.squeeze()
    for i_f in range(n_calibration):
        T_epoch_LOO[0,i_f] = one_complex_ada_size(dist_cal, i_f, -1, epoch_ycal, k, Tmax, Tmin, p)
        for y_js in js_y_all:
            T_epoch[y_js, i_f] = one_complex_ada_size(dist_all, i_f, y_js, epoch_ycal, k, Tmax, Tmin, p)
    return T_epoch_pointtest, T_epoch, T_epoch_LOO

def many_complex_ada_w(n_class, n_calibration, sorted_one_epoch, T_epoch_pointtest, T_epoch, T_epoch_LOO):
    js_y_all = torch.tensor(range(n_class), device=device)
    w_epoch = torch.zeros(n_class, n_calibration, device=device)
    w_epoch_LOO = torch.zeros(1, n_calibration, device=device)
    w_epoch_pointtest = sorted_one_epoch[0,T_epoch_pointtest]
    for i_f in range(n_calibration):
        w_epoch_LOO[0,i_f] = sorted_one_epoch[i_f+1,T_epoch_LOO[0, i_f] ]
        for y_js in js_y_all:
            w_epoch[y_js, i_f] = sorted_one_epoch[i_f+1, T_epoch[y_js, i_f]]
    return w_epoch_pointtest, w_epoch, w_epoch_LOO

def alpha_compute(true_S_cal, w_cal, w_test, n_calibration, epoch):
    LOO_Ssum = true_S_cal.sum(dim=1, keepdim=True) - true_S_cal  # (epoch, n_calibration)
    alpha_LOO_i = (1 / n_calibration) * (LOO_Ssum / w_cal + 1)  # (epoch, n_calibration)
    alpha_LOO_i = torch.clamp(alpha_LOO_i, min=0.0, max=1)
    alpha_LOO = alpha_LOO_i.mean(dim=1, keepdim=True)  # (epoch, 1)
    alpha_LOOnp = alpha_LOO.cpu().numpy().squeeze()  # (epoch,)
    alpha_real = (((true_S_cal.sum(dim=1) / w_test.squeeze() + 1)/(n_calibration + 1)).sum(dim=0))/epoch
    alpha_realnp = alpha_real.cpu().numpy()
    return alpha_LOOnp, alpha_realnp

def cor_alpha_compute(true_S_cal, true_S_test, w_cal, w_test, n_calibration, epoch):
    LOO_Ssum = true_S_cal.sum(dim=1, keepdim=True) - true_S_cal  # (epoch, n_calibration)
    E_test = (n_calibration+1)*true_S_test / true_S_cal.sum(dim=1, keepdim=True)
    E_LOO = n_calibration*true_S_cal / LOO_Ssum
    alpha_LOO_i = (1 / n_calibration) * (LOO_Ssum / w_cal + 1)  # (epoch, n_calibration)
    cor_alpha_LOO_i = E_LOO* alpha_LOO_i
    cor_alpha_LOO_i = torch.clamp(cor_alpha_LOO_i, min=0.0, max=1)
    cor_alpha_LOO = cor_alpha_LOO_i.mean(dim=1, keepdim=True)  # (epoch, 1)
    cor_alpha_LOOnp = cor_alpha_LOO.cpu().numpy().squeeze()
    cor_alpha_real = ((E_test.squeeze(1)*((true_S_cal.sum(dim=1) / w_test.squeeze(1) + 1)/(n_calibration + 1))).sum(dim=0))/epoch
    cor_alpha_real = torch.clamp(cor_alpha_real, min=0.0, max=1)
    cor_alpha_realnp = cor_alpha_real.cpu().numpy()
    alpha_LOO_i = torch.clamp(alpha_LOO_i, min=0.0, max=1)
    alpha_LOO = alpha_LOO_i.mean(dim=1, keepdim=True)  # (epoch, 1)
    alpha_LOOnp = alpha_LOO.cpu().numpy().squeeze()  # (epoch,)
    alpha_real = (((true_S_cal.sum(dim=1) / w_test.squeeze(1) + 1)/(n_calibration + 1)).sum(dim=0))/epoch
    alpha_realnp = alpha_real.cpu().numpy()
    return alpha_LOOnp, alpha_realnp, cor_alpha_LOOnp, cor_alpha_realnp

def ro_alpha_compute(true_S_cal, n_calibration, epoch):
    LOO_Ssum = true_S_cal.sum(dim=1, keepdim=True) - true_S_cal
    alpha_LOO_i = (1 / n_calibration) * (LOO_Ssum + 1)
    alpha_LOO_i = torch.clamp(alpha_LOO_i, min=0.0, max=1)
    alpha_LOO = alpha_LOO_i.mean(dim=1, keepdim=True)
    alpha_LOOnp = alpha_LOO.cpu().numpy().squeeze()
    alpha_real = (((true_S_cal.sum(dim=1) + 1)/(n_calibration + 1)).sum(dim=0))/epoch
    alpha_realnp = alpha_real.cpu().numpy()
    return alpha_LOOnp, alpha_realnp

def fun_TC_C(C, true_S_cal, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch):
    T_t2_calibration = torch.full((epoch, n_calibration), C, dtype=torch.long, device=device)
    T_t2_point_test = torch.full((epoch, 1), C, dtype=torch.long, device=device)
    w_t2_calibration = torch.gather(sorted_S_t3_cal, dim=2, index=T_t2_calibration.unsqueeze(2)).squeeze(2)
    w_t2_point_test = torch.gather(sorted_S_t3_test, dim=2, index=T_t2_point_test.unsqueeze(2)).squeeze(2)
    true_tr_S_t2_cal = w_t2_calibration*((true_r_cal>T_t2_calibration).float())+1e-4
    re_pro = ((true_r_test>C).sum(dim=0))/epoch
    re_pronp = re_pro.cpu().numpy()
    or_alpha_LOOnp, or_alpha_realnp = alpha_compute(true_S_cal, w_t2_calibration, 
                                                    w_t2_point_test, n_calibration, epoch)
    tr_alpha_LOOnp, tr_alpha_realnp = alpha_compute(true_tr_S_t2_cal, w_t2_calibration,
                                                    w_t2_point_test, n_calibration, epoch)
    return or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp

def cor_fun_TC_C(C, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch):
    T_t2_calibration = torch.full((epoch, n_calibration), C, dtype=torch.long, device=device)
    T_t2_point_test = torch.full((epoch, 1), C, dtype=torch.long, device=device)
    w_t2_calibration = torch.gather(sorted_S_t3_cal, dim=2, index=T_t2_calibration.unsqueeze(2)).squeeze(2)
    w_t2_point_test = torch.gather(sorted_S_t3_test, dim=2, index=T_t2_point_test.unsqueeze(2)).squeeze(2)
    ture_tr_S_t2_cal = w_t2_calibration*((true_r_cal>T_t2_calibration).float())+1e-4
    ture_tr_S_test = w_t2_point_test*((true_r_test.unsqueeze(1)>T_t2_point_test).float())+1e-4
    re_pro = ((true_r_test>C).sum(dim=0))/epoch
    re_pronp = re_pro.cpu().numpy()
    tr_alpha_LOOnp, tr_alpha_realnp, cor_tr_alpha_LOOnp, cor_tr_alpha_realnp = cor_alpha_compute(
        ture_tr_S_t2_cal, ture_tr_S_test, w_t2_calibration, w_t2_point_test, n_calibration, epoch)
    return tr_alpha_LOOnp, tr_alpha_realnp, cor_tr_alpha_LOOnp, cor_tr_alpha_realnp, re_pronp

def ro_fun_TC_C(C, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch):
    T_t2_calibration = torch.full((epoch, n_calibration), C, dtype=torch.long, device=device)
    T_t2_point_test = torch.full((epoch, 1), C, dtype=torch.long, device=device)
    w_t2_calibration = torch.gather(sorted_S_t3_cal, dim=2, index=T_t2_calibration.unsqueeze(2)).squeeze(2)
    w_t2_point_test = torch.gather(sorted_S_t3_test, dim=2, index=T_t2_point_test.unsqueeze(2)).squeeze(2)
    true_tr_S_t2_cal = w_t2_calibration*((true_r_cal>T_t2_calibration).float())+1e-4
    re_pro = ((true_r_test>C).sum(dim=0))/epoch
    re_pronp = re_pro.cpu().numpy()
    tr_alpha_LOOnp, tr_alpha_realnp = alpha_compute(true_tr_S_t2_cal, w_t2_calibration,
                                                    w_t2_point_test, n_calibration, epoch)
    ro_true_tr_S_t2_cal = (true_r_cal>T_t2_calibration).float()+1e-4
    ro_alpha_LOOnp, ro_alpha_realnp = ro_alpha_compute(ro_true_tr_S_t2_cal, n_calibration, epoch)
    return tr_alpha_LOOnp, tr_alpha_realnp, ro_alpha_LOOnp, ro_alpha_realnp, re_pronp

def plot_lines(x_data, y_data1, y_data2, name='MSE', flag=True, save=False):
    plt.plot(x_data, y_data1, linewidth = 2, color='#1f77b4')
    plt.plot(x_data, y_data2, linewidth = 2, color='#ff7f0e')
    plt.scatter(x_data, y_data1, color='#1f77b4', marker='o') 
    plt.scatter(x_data, y_data2, color='#ff7f0e', marker='s') 
    plt.xlabel('n')
    plt.ylabel(name, fontsize=12)
    plt.grid(axis='y', linestyle='--')

    if flag:
        legend_elements = [
            Line2D([0], [0], color='#1f77b4', lw=2, label=r'BCP', marker='o', markersize=6),
            Line2D([0], [0], color='#ff7f0e', lw=2, label=r'ST-BCP(ours)', marker='s', markersize=6)]
        plt.legend(handles=legend_elements, fontsize=14)
    if save:
        save_path = os.path.join('fig', 'line'+name+'.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
def plot_bar(x_data, y_data1, y_data2, name = 'MSE', flag=True, save=False):
    bar_width = 0.35
    index = np.arange(len(x_data))
    plt.bar(index - bar_width/2, y_data1, bar_width, color='#1f77b4', label=r'BCP')
    plt.bar(index + bar_width/2, y_data2, bar_width, color='#ff7f0e', label=r'ST-BCP(ours)')
    plt.ylabel(name, fontsize=12)
    plt.xticks(index, x_data)
    if flag:
        plt.legend(fontsize=12)
    if save:
        save_path = os.path.join('fig', 'bar'+name+'.pdf')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
def plot_density_with_lines(y1, y2, a1, a2, a3, dataset_name, flag=True, save=False):
    bw_adjust=1.2
    sns.kdeplot(y1, label=r'$\hat{\alpha}^{LOO}(s)$', color='blue', fill = True, bw_adjust=bw_adjust)
    sns.kdeplot(y2, label=r'$\hat{\alpha}^{LOO}(I_{w})$', color='green', fill = True, bw_adjust=bw_adjust)
    plt.axvline(x=a1, color='blue', linestyle='--', linewidth = 3, label=r'$E[\tilde{\alpha}(s)]$')
    plt.axvline(x=a2, color='green', linestyle='--', linewidth = 3, label=r'$E[\tilde{\alpha}(I_{w})]$')
    plt.axvline(x=a3, color='red', linestyle='--', linewidth = 3, label=r'$P(Y_{n+1}\notin C(X_{n+1}))$')
    plt.xlabel(r'$\alpha$')
    plt.ylabel('Density', fontsize=12)
    if flag:
        plt.legend()
    if save:
        save_path = os.path.join('fig', dataset_name+'_density.pdf')
        plt.savefig(save_path, dpi=300)
    plt.show()
    
def plot_cor_density_with_lines(kon, y1, y2, a1, a2, a3, dataset_name, flag=True, save=False):
    bw_adjust=1.2
    sns.kdeplot(y2, label=r'$\hat{b}^{LOO}(I_{w})$', color='#666666', fill = True, bw_adjust=bw_adjust)
    sns.kdeplot(y1, label=r'$\hat{\alpha}^{LOO}(I_{w})$', color='green', fill = True, bw_adjust=bw_adjust)
    plt.axvline(x=a2, color='#666666', linestyle='--', linewidth = 3, label=r'$E[\tilde{b}(I_{w})]$')
    plt.axvline(x=a1, color='green', linestyle='--', linewidth = 3, label=r'$E[\tilde{\alpha}(I_{w})]$')
    plt.axvline(x=a3, color='red', linestyle='--', linewidth = 3, label=r'$P(Y_{n+1}\notin C(X_{n+1}))$')
    plt.xlabel(r'$\alpha$')
    plt.ylabel('Density', fontsize=12)
    if flag:
        plt.legend()
    if save:
        save_path = os.path.join('fig', kon + 'cor_' + dataset_name + '_density.pdf')
        plt.savefig(save_path, dpi=300)
    plt.show()

def plot_cor_density_with_lines(kon, y1, y2, a1, a2, a3, dataset_name, flag=True, save=False):
    bw_adjust=1.2
    sns.kdeplot(y2, label=r'$\hat{\alpha}^{LOO}(I_{ro})$', color='#189ED7', fill = True, bw_adjust=bw_adjust)
    sns.kdeplot(y1, label=r'$\hat{\alpha}^{LOO}(I_{w})$', color='green', fill = True, bw_adjust=bw_adjust)
    plt.axvline(x=a2, color="#189ED7", linestyle='--', linewidth = 3, label=r'$E[\tilde{\alpha}(I_{ro})]$')
    plt.axvline(x=a1, color='green', linestyle='--', linewidth = 3, label=r'$E[\tilde{\alpha}(I_{w})]$')
    plt.axvline(x=a3, color='red', linestyle='--', linewidth = 3, label=r'$P(Y_{n+1}\notin C(X_{n+1}))$')
    plt.xlabel(r'$\alpha$')
    plt.ylabel('Density', fontsize=12)
    if flag:
        plt.legend()
    if save:
        save_path = os.path.join('fig', kon + 'ro_' + dataset_name + '_density.pdf')
        plt.savefig(save_path, dpi=300)
    plt.show()


In [ ]:
#Calculate the non-conformity score
#You need to train resnet50 in CIFAR10 using train_main.ipynb
dataset_name = 'CIFAR100'
model_name = 'resnet50'
n_calibration = 200
n_pici = 500
epoch = 500
n_ml = epoch // n_pici
testset, C_data, H_data, W_data = select_dataset(dataset_name)
params = dataset_params[dataset_name]
n_class = params['num_classes']
net = select_model(model_name, dataset_name, device)
S_t3_calibration, S_t3_point_test, y_all, memory_idx = funS_t3(
    net, testset, n_calibration, n_pici, n_ml, n_class, 
    C_data, H_data, W_data, epoch, device)
r_t3_calibration, r_t3_point_test = funr_t3(S_t3_calibration, S_t3_point_test)
true_S_cal, true_S_test, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test = funture_rS(
        S_t3_calibration, S_t3_point_test, r_t3_calibration, r_t3_point_test, y_all)

In [ ]:
#tabel:T(D,X)
#You need to calculate the non-conformity score
#You need to download the pre-train weights of resnet18, 
#https://download.pytorch.org/models/resnet18-f37072fd.pth
#You need to change "your_resnet18_path" by your own path.
Tmin = params["Tmin_knn"]
Tmax = params["Tmax_knn"]
k = params["k_knn"]
p = params["p_knn"]
eif = 0.01

weights_path_res = "your_resnet18_path"
net_res = resnet18(weights = None)
net_res.load_state_dict(torch.load(weights_path_res, map_location=device))
net_res.fc = nn.Identity()
if device == 'cuda':
    net_res = torch.nn.DataParallel(net_res)
    cudnn.benchmark = True
y_point_test = y_all[:, 0]  # (epoch,)
y_calibration = y_all[:, 1:]  # (epoch, n_calibration)
T_all_pointtest = torch.zeros(epoch, device=device)
T_all = torch.zeros(epoch, n_class ,n_calibration, device=device)
T_all_LOO = torch.zeros(epoch, 1, n_calibration, device=device)
w_all_pointtest = torch.zeros(epoch, device=device)
w_all = torch.zeros(epoch, n_class ,n_calibration, device=device)
w_all_LOO = torch.zeros(epoch, 1, n_calibration, device=device)
epo_i = 0
js_y_all = torch.tensor(range(n_class), device=device)
net_res.eval()
with torch.no_grad():
    X_test_all, y_test_all = next(iter(DataLoader(testset, batch_size=len(testset))))
    X_test_all, y_test_all = X_test_all.to(device), y_test_all.to(device)
    for i in range(n_pici):
        idx = memory_idx[i]  # (n_ml, n_calibration + 1)
        X_idx = X_test_all[idx]  # (n_ml, n_calibration + 1, channels, height, width)
        X_idx = X_idx.view(-1, C_data, H_data, W_data)
        y_idx = y_test_all[idx]  # (n_ml, n_calibration + 1)
        logits_res_idx = net_res(X_idx)
        logits_res_idx = logits_res_idx.view(n_ml, n_calibration + 1, 512) # (n_ml, n_calibration + 1, 512)
        for epoch_idx in range(n_ml):
            epoch_reslogits = logits_res_idx[epoch_idx] # (n_calibration + 1, 512)
            epoch_reslogits_cal = epoch_reslogits[1:]
            U, _, _ = torch.pca_lowrank(epoch_reslogits, q=2) #(n_calibration+1, 2)
            U_cal,_,_ = torch.pca_lowrank(epoch_reslogits_cal, q=2)
            dist_all_matrix = torch.cdist(U, U, p = 2) #(n_calibration+1,n_calibration+1)
            dist_cal_matrix = torch.cdist(U_cal, U_cal, p = 2)
            epoch_y = y_idx[epoch_idx]  # (n_calibration + 1)
            epoch_ycal = epoch_y[1:]
            sorted_one_epoch = torch.cat([sorted_S_t3_test[epo_i,:,:], 
                              sorted_S_t3_cal[epo_i,:,:].squeeze(0)],dim=0)
            T_epoch_pointtest, T_epoch, T_epoch_LOO=many_complex_ada_size(dist_all_matrix, dist_cal_matrix, epoch_ycal, n_class, 
                                                       n_calibration, k, Tmax, Tmin, p)
            w_epoch_pointtest, w_epoch, w_epoch_LOO=many_complex_ada_w(n_class, n_calibration, 
                                        sorted_one_epoch,T_epoch_pointtest.long(), T_epoch.long(), T_epoch_LOO.long())
            T_all_pointtest[epo_i] = T_epoch_pointtest
            T_all[epo_i,:,:] = T_epoch
            T_all_LOO[epo_i,:,:] = T_epoch_LOO
            w_all_pointtest[epo_i] = w_epoch_pointtest
            w_all[epo_i,:,:] = w_epoch
            w_all_LOO[epo_i,:,:] = w_epoch_LOO
            epo_i = epo_i + 1


ture_tr_S_cal_LOO = (w_all_LOO.squeeze(1))*((true_r_cal>(T_all_LOO.squeeze(1))).float())
repeat_true_S_cal = true_S_cal.unsqueeze(1).expand(epoch, n_class, n_calibration)
tr_jsy_S_cal = w_all*fh(repeat_true_S_cal, w_all, eif)
tr_jsy_sum_Scal = tr_jsy_S_cal.sum(dim=2)
repeat_w_point_test = w_all_pointtest.unsqueeze(1).expand(epoch, n_class)
tr_S_point_test = repeat_w_point_test*fh(S_t3_point_test.squeeze(1), repeat_w_point_test, eif)
tr_E_point_test = (n_calibration+1)*(tr_S_point_test/(tr_S_point_test+tr_jsy_sum_Scal))
sorted_trE_point_test,_ = torch.sort(tr_E_point_test,dim=1,descending=False)
r_E_point_test = tr_E_point_test.argsort(dim=1).argsort(dim=1) + 1
true_r_E_point_test = r_E_point_test.gather(1, y_point_test.unsqueeze(1)).squeeze(1)
tr_alpha_LOO, tr_alpha_real = alpha_compute(
    true_S_cal, w_all_LOO.squeeze(1), w_all_pointtest, n_calibration, epoch)

tr_LOO_Ssum = ture_tr_S_cal_LOO.sum(dim=1, keepdim=True) - ture_tr_S_cal_LOO  # (epoch, n_calibration)
tr_alpha_LOO_i = (1 / n_calibration) * (tr_LOO_Ssum / (w_all_LOO.squeeze(1)) + 1)  # (epoch, n_calibration)
tr_alpha_LOO_i = torch.clamp(tr_alpha_LOO_i, min=0.0, max=1)
tr_alpha_LOO = tr_alpha_LOO_i.mean(dim=1, keepdim=True)  # (epoch, 1)
tr_alpha_LOOnp = tr_alpha_LOO.cpu().numpy().squeeze()  # (epoch,)
tr_alpha_real = 1/(sorted_trE_point_test.gather(dim=1, index=(T_all_pointtest.long()).unsqueeze(1)).squeeze(1))
tr_alpha_real = tr_alpha_real.sum(dim=0)/epoch
tr_alpha_realnp = tr_alpha_real.cpu().numpy()
tr_misvoc = (((true_r_E_point_test>T_all_pointtest).long()).sum(dim=0))/epoch
re_pro_E = tr_misvoc.cpu().numpy()

or_alpha_LOOnp, or_alpha_realnp = alpha_compute(
    true_S_cal, w_all_LOO.squeeze(1), w_all_pointtest, n_calibration, epoch)
or_miscov = (((true_r_test>T_all_pointtest).long()).sum(dim=0))/epoch
re_pronp = or_miscov.cpu().numpy()

a_an(or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp)
print(f"The accurate coverage under the correlation of transformation and calibration set: {re_pro_E:.3g}")
            

In [ ]:
#tabel1:T(C),C=1,2,3
#You need to calculate the non-conformity score
Tmin = params["Tmin_C"]
Tmax = params["Tmax_C"]
for C in np.arange(Tmin, Tmax+1, 1):
    or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp = fun_TC_C(
        C, true_S_cal, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch)
    a_an(or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp)


In [ ]:
#tabel1:T(X)
#You need to calculate the non-conformity score
Tmax = params["Tmax_f"]
Tmin = params["Tmin_f"]
p = params["p_f"]

pro_t3_calibration = torch.exp(-S_t3_calibration)
pro_t3_point_test = torch.exp(-S_t3_point_test)
enmo_t2_calibration = (S_t3_calibration*pro_t3_calibration).sum(dim=2)
enmo_t2_point_test = (S_t3_point_test*pro_t3_point_test).sum(dim=2)
enmo_t2 = torch.cat([enmo_t2_calibration, enmo_t2_point_test], dim=1)  # (epoch, n_calibration + 1)

bin = bin_f_thresholds(enmo_t2, n_class, Tmax=Tmax, Tmin=Tmin, p=p)
T_t2 = T_ad(enmo_t2, Tmax=Tmax, Tmin=Tmin, p=p, bin=bin)
T_t2 = T_t2.long()  # (epoch, n_calibration + 1)
T_t2_point_test = T_t2[:,0]
T_t2_point_test = T_t2_point_test.unsqueeze(1)  # (epoch, 1)
T_t2_calibration = T_t2[:,1:]

re_pro = ((true_r_test>T_t2_point_test.squeeze(1)).sum(dim=0))/epoch
re_pronp = re_pro.cpu().numpy()

w_t2_calibration = torch.gather(sorted_S_t3_cal, dim=2, index=T_t2_calibration.unsqueeze(2)).squeeze(2)
w_t2_point_test = torch.gather(sorted_S_t3_test, dim=2, index=T_t2_point_test.unsqueeze(2)).squeeze(2)
ture_tr_S_t2_cal = w_t2_calibration*((true_r_cal>T_t2_calibration).float())
or_alpha_LOOnp, or_alpha_realnp = alpha_compute(true_S_cal, w_t2_calibration, 
                                                    w_t2_point_test, n_calibration, epoch)
tr_alpha_LOOnp, tr_alpha_realnp = alpha_compute(ture_tr_S_t2_cal, w_t2_calibration, 
                                                    w_t2_point_test, n_calibration, epoch)
a_an(or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp)

In [ ]:
#figure2: (a) and (b)
#You need to train resnet50 in CIFAR10 using train_main.ipynb
dataset_name = 'CIFAR10'
model_name = 'resnet50'
n_pici = 250
epoch = 500
n_ml = epoch // n_pici
testset, C_data, H_data, W_data = select_dataset(dataset_name)
params = dataset_params[dataset_name]
n_class = params['num_classes']
net = select_model(model_name, dataset_name, device)
C = 2

list_or_std_LOOnp = []
list_or_mse = []
list_tr_std_LOOnp = []
list_tr_mse = []
list_n_calibration = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
for n_calibration in list_n_calibration:
    S_t3_calibration, S_t3_point_test, y_all, memory_idx = funS_t3(
        net, testset, n_calibration,n_pici, n_ml, n_class, C_data, H_data, W_data, epoch, device)
    r_t3_calibration, r_t3_point_test = funr_t3(S_t3_calibration, S_t3_point_test)
    true_S_cal, true_S_test, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test = funture_rS(
        S_t3_calibration, S_t3_point_test, r_t3_calibration, r_t3_point_test, y_all)
    or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp = fun_TC_C(
        C, true_S_cal, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch)
    or_std_LOOnp = np.std(or_alpha_LOOnp)
    or_mse = np.mean((or_alpha_LOOnp - or_alpha_realnp) ** 2)
    tr_std_LOOnp = np.std(tr_alpha_LOOnp)
    tr_mse = np.mean((tr_alpha_LOOnp - tr_alpha_realnp) ** 2)
    list_or_std_LOOnp.append(or_std_LOOnp)
    list_or_mse.append(or_mse)
    list_tr_std_LOOnp.append(tr_std_LOOnp)
    list_tr_mse.append(tr_mse)

plot_lines(list_n_calibration, list_or_std_LOOnp, list_tr_std_LOOnp, name='Std', save=True)
plot_lines(list_n_calibration, list_or_mse, list_tr_mse, name='MSE', save=True)

In [ ]:
#figure2: (a) and (b)
#You need to train fours models in CIFAR10 using train_main.ipynb
dataset_name = 'CIFAR10'
n_calibration = 200
n_pici = 250
epoch = 500
n_ml = epoch // n_pici
testset, C_data, H_data, W_data = select_dataset(dataset_name)
params = dataset_params[dataset_name]
n_class = params['num_classes']
C = 2

list_or_mse = []
list_tr_mse = []
list_or_gap = []
list_tr_gap = []
list_model_name = ['mobilenetv3small', 'efficientnetb0', 'resnet50', 'densenet121']
for model_name in list_model_name:
    net = select_model(model_name, dataset_name, device)
    S_t3_calibration, S_t3_point_test, y_all, memory_idx = funS_t3(
        net, testset, n_calibration, n_pici, n_ml, n_class, C_data, H_data, W_data, epoch, device)
    r_t3_calibration, r_t3_point_test = funr_t3(S_t3_calibration, S_t3_point_test)
    true_S_cal, true_S_test, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test = funture_rS(
        S_t3_calibration, S_t3_point_test, r_t3_calibration, r_t3_point_test, y_all)
    or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp = fun_TC_C(
        C, true_S_cal, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch)
    or_mse = np.mean((or_alpha_LOOnp - or_alpha_realnp) ** 2)
    or_gap = np.mean(np.abs(or_alpha_LOOnp - re_pronp))
    tr_mse = np.mean((tr_alpha_LOOnp - tr_alpha_realnp) ** 2)
    tr_gap = np.mean(np.abs(tr_alpha_LOOnp - re_pronp))
    list_or_mse.append(or_mse)
    list_or_gap.append(or_gap)
    list_tr_mse.append(tr_mse)
    list_tr_gap.append(tr_gap)

plot_bar(list_model_name, list_or_mse, list_tr_mse, name='MSE', save=False)
plot_bar(list_model_name, list_or_gap, list_tr_gap, name='GAP', save=False)


In [ ]:
##figure1: (a), (b) and (c)
#You need to train resnet50 in three datasets using train_main.ipynb
n_calibration = 200
n_pici = 250
epoch = 500
n_ml = epoch // n_pici
model_name = 'resnet50'
flag = True
list_dataset_name = ['CIFAR10', 'CIFAR100', 'TinyImageNet']
for dataset_name in list_dataset_name:
    params = dataset_params[dataset_name]
    C = params['ad_C']
    n_class = params['num_classes']
    testset, C_data, H_data, W_data = select_dataset(dataset_name)
    net = select_model(model_name, dataset_name, device)
    S_t3_calibration, S_t3_point_test, y_all, memory_idx = funS_t3(
        net, testset, n_calibration, n_pici, n_ml, n_class, 
        C_data, H_data, W_data, epoch, device)
    r_t3_calibration, r_t3_point_test = funr_t3(S_t3_calibration, S_t3_point_test)
    true_S_cal, true_S_test, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test = funture_rS(
        S_t3_calibration, S_t3_point_test, r_t3_calibration, r_t3_point_test, y_all)
    or_alpha_LOOnp, or_alpha_realnp, tr_alpha_LOOnp, tr_alpha_realnp, re_pronp = fun_TC_C(
        C, true_S_cal, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch)
    plot_density_with_lines(or_alpha_LOOnp, tr_alpha_LOOnp, or_alpha_realnp, tr_alpha_realnp, re_pronp, flag=flag, save=False, dataset_name=dataset_name)
    flag = False

In [ ]:
#figure4
#You need to train resnet50 in three datasets using train_main.ipynb
n_calibration = 200
n_pici = 250
epoch = 500
n_ml = epoch // n_pici
model_name = 'resnet50'
C_fix = 1
flag = False
list_dataset_name = ['CIFAR10', 'CIFAR100', 'TinyImageNet']
for dataset_name in list_dataset_name:
    params = dataset_params[dataset_name]
    C_ad = params['ad_C']
    params = dataset_params[dataset_name]
    n_class = params['num_classes']
    testset, C_data, H_data, W_data = select_dataset(dataset_name)
    net = select_model(model_name, dataset_name, device)
    S_t3_calibration, S_t3_point_test, y_all, memory_idx = funS_t3(
        net, testset, n_calibration, n_pici, n_ml, n_class, 
        C_data, H_data, W_data, epoch, device)
    r_t3_calibration, r_t3_point_test = funr_t3(S_t3_calibration, S_t3_point_test)
    true_S_cal, true_S_test, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test = funture_rS(
        S_t3_calibration, S_t3_point_test, r_t3_calibration, r_t3_point_test, y_all)
    tr_alpha_LOOnp, tr_alpha_realnp, cor_tr_alpha_LOOnp, cor_tr_alpha_realnp, re_pronp = cor_fun_TC_C(
    C_fix, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch)
    plot_cor_density_with_lines("fix_", tr_alpha_LOOnp, cor_tr_alpha_LOOnp, tr_alpha_realnp, 
                                cor_tr_alpha_realnp, re_pronp, flag=flag, save=True, 
                                dataset_name=dataset_name)
    flag = False
    tr_alpha_LOOnp, tr_alpha_realnp, cor_tr_alpha_LOOnp, cor_tr_alpha_realnp, re_pronp = cor_fun_TC_C(
    C_ad, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch)
    plot_cor_density_with_lines("ad_", tr_alpha_LOOnp, cor_tr_alpha_LOOnp, tr_alpha_realnp, 
                                cor_tr_alpha_realnp, re_pronp, flag=flag, save=True, 
                                dataset_name=dataset_name)   
    

In [ ]:
#figure5
#You need to train resnet50 in three datasets using train_main.ipynb
n_calibration = 200
n_pici = 250
epoch = 500
n_ml = epoch // n_pici
model_name = 'resnet50'
C_fix = 1
flag = True
list_dataset_name = ['CIFAR10', 'CIFAR100', 'TinyImageNet']
for dataset_name in list_dataset_name:
    params = dataset_params[dataset_name]
    C_ad = params['ad_C']
    params = dataset_params[dataset_name]
    n_class = params['num_classes']
    testset, C_data, H_data, W_data = select_dataset(dataset_name)
    net = select_model(model_name, dataset_name, device)
    S_t3_calibration, S_t3_point_test, y_all, memory_idx = funS_t3(
        net, testset, n_calibration, n_pici, n_ml, n_class, 
        C_data, H_data, W_data, epoch, device)
    r_t3_calibration, r_t3_point_test = funr_t3(S_t3_calibration, S_t3_point_test)
    true_S_cal, true_S_test, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test = funture_rS(
        S_t3_calibration, S_t3_point_test, r_t3_calibration, r_t3_point_test, y_all)
    tr_alpha_LOOnp, tr_alpha_realnp, cor_tr_alpha_LOOnp, cor_tr_alpha_realnp, re_pronp = cor_fun_TC_C(
    C_fix, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch)
    plot_cor_density_with_lines("fix_", tr_alpha_LOOnp, cor_tr_alpha_LOOnp, tr_alpha_realnp, 
                                cor_tr_alpha_realnp, re_pronp, flag=flag, save=True, 
                                dataset_name=dataset_name)
    flag = False
    tr_alpha_LOOnp, tr_alpha_realnp, ro_alpha_LOOnp, ro_alpha_realnp, re_pronp=ro_fun_TC_C(
    C_ad, true_r_cal, true_r_test, sorted_S_t3_cal, sorted_S_t3_test, n_calibration, epoch)
    plot_cor_density_with_lines("ad_", tr_alpha_LOOnp, ro_alpha_LOOnp, tr_alpha_realnp, 
                                ro_alpha_realnp , re_pronp, flag=flag, save=True, 
                                dataset_name=dataset_name) 
    